# Module 05 — SAC on continuous control

Output real-valued actions — the regime real robots live in. See [the lesson](README.md).

> Tip: in Colab, set **Runtime → Change runtime type → GPU**

In [ ]:
# === Colab setup: run me first ===
import os, sys
if not os.path.exists('rl'):
    # On Colab, clone the repo so the `rl` package is importable.
    !git clone https://github.com/anhduckkzz/lunarlander.git repo && (cp -r repo/* . 2>/dev/null || true)
    !pip -q install 'gymnasium[box2d]>=0.29' torch numpy matplotlib imageio tqdm
import torch
print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## Notebook type and ordered study structure

**Notebook type:** Type A — core mechanism notebook. This should be studied slowly because the mechanism appears again and again across the whole repo.

**Your objective in this notebook:** Study continuous actions for robotics using SAC/DDPG/TD3 ideas.

Use this exact order every time:

1. **Why this matters** — identify the real problem this topic solves.
2. **Mental model** — explain the idea in plain language before symbols.
3. **Math mechanism** — write the smallest formula and define every symbol.
4. **From-scratch code** — run/read the simple implementation slowly.
5. **Line-by-line explanation** — trace inputs, internal variables, update rule, and outputs.
6. **Debug/visualize** — print shapes, values, curves, maps, or weights so behavior is visible.
7. **Framework version** — map the mechanism to a real library/API.
8. **Scratch → framework mapping** — write what the framework hides and what it exposes.
9. **Real-system role** — place the topic inside robotics, driving, drones, manipulation, or VLA.
10. **Failure modes** — list how it breaks and how you would notice.
11. **Exercises** — change parameters, break the example, and explain the result.
12. **Mini-project** — build a small artifact you can keep.
13. **Next step** — choose the next notebook or tool.

**Mental model for this topic:** Robots often output torques, velocities, or position deltas, not discrete actions.

**Core math / mechanism to keep in mind:** SAC maximizes reward + alpha * entropy to stay exploratory.

**Recommended debug habit:** after every code cell, ask “what are the inputs, what changed, and what would be unsafe or wrong in a real robot?”

## From-scratch focus and code-reading checklist

**Scratch focus:** Run/read SAC on continuous LunarLander.

When reading code in this notebook or the matching repo module, trace it like this:

| Step | Question to answer |
|---|---|
| Input | What is the state, observation, tensor, point, reward, or measurement? |
| Representation | Is it a scalar, vector, matrix, image, point cloud, token sequence, or action? |
| Mechanism | Which line implements the math/update rule? |
| Parameters | Which numbers are hyperparameters, physical constants, or learned weights? |
| Output | What changed after the step? |
| Debug signal | What should I print/plot to know it is working? |

Do not move to the framework/API version until you can explain the scratch version without reading the code comments.

## Visualization and debugging ideas

Use at least one of these while studying:

- Print tensor/vector shapes before and after the core operation.
- Print the first few values before and after an update.
- Plot a curve when there is learning, control, filtering, planning, or optimization.
- Draw frames, maps, paths, sensor rays, or attention matrices when geometry is involved.
- Change one parameter at a time and predict the effect before running.

For this notebook, a useful first visualization/debug target is: **Run/read SAC on continuous LunarLander.**

## Scratch → framework mapping

| From-scratch idea in this repo | Practical framework/API | What to learn from the framework |
|---|---|---|
| SquashedGaussianActor | SB3 SAC policy | tanh-squashed actions keep outputs within action limits |
| ContinuousQNetwork | twin critics | SAC/TD3 reduce overestimation |
| action_scale | env.action_space | frameworks normalize and rescale actions |

**Framework learning rule:** do not memorize the API first. First identify which scratch concept it replaces, then learn its inputs, outputs, configuration, and failure modes.

## Real-system application

Continuous control is central for arms, drones, quadrupeds, and vehicle control.

Ask these system questions:

1. What module produces the input to this component?
2. What module consumes its output?
3. What latency, safety, calibration, or data-format assumptions exist?
4. What metric tells me this component is good enough for the larger system?

## Failure modes and debugging

Common ways this topic can fail:

- action saturation
- unsafe exploration on hardware
- entropy too high/low
- sim action scaling mismatch

For each failure, write:

- **Symptom:** what would I see in logs, plots, robot behavior, or evaluation?
- **Likely cause:** what assumption broke?
- **First debug action:** what is the smallest thing to inspect?

## Mini-project and mastery checklist

**Mini-project:** Compare action ranges and normalization for LunarLanderContinuous vs a robot arm joint controller.

Mastery checklist:

- [ ] I can explain the mental model in one paragraph.
- [ ] I can write the core formula and define every symbol.
- [ ] I can run or read the scratch code and point to the core update/operation.
- [ ] I can name the production framework/API version of the same idea.
- [ ] I can describe where this topic sits in a robot/car/drone/VLA stack.
- [ ] I can name at least three failure modes and one debug action for each.

**Next study steps:** Part 14 control, Part 17 locomotion, RL 15 safe sim-to-real

## SAC on continuous LunarLander

In [ ]:
from rl.envs import make_env, env_dims
from rl.agents.sac import SAC, SACConfig
from rl.train import train_offpolicy
from rl.utils import Logger, plot_scores, set_seed

set_seed(0)
env = make_env('LunarLander-v3', continuous=True, seed=0)
s_dim, a_dim, _ = env_dims(env)
scale = float(env.action_space.high[0])
agent = SAC(s_dim, a_dim, action_scale=scale,
            cfg=SACConfig(learning_starts=5000), device=DEVICE)
scores = train_offpolicy(agent, env, n_steps=200_000, discrete=False,
                         logger=Logger('runs/sac_nb'), solved_at=200)
plot_scores(scores, title='SAC on LunarLanderContinuous', show=True)

**Compare:** swap `SAC` for `TD3` and `DDPG`. Plot all three on the same axes to feel the sample-efficiency and stability differences.